In [0]:
from pyspark.sql import functions as F
catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]
spark.sql("SELECT current_catalog()").show()
spark.sql("SHOW CATALOGS").show()
schema = "migration"
volume = "urg_migration"
base_path = f"/Volumes/{catalog}/{schema}/{volume}"
files={
    "Customers":"customers_legacy_large.csv",
    "Orders":"orders_legacy_large.csv",
    "Products":"products_legacy_large.csv",
    "Transactions":"transactions_legacy_large.csv",
}

bronze = f"{catalog}.{schema}.'bronze_data'"

In [0]:
for name,file_name in files.items():
    path = f"{base_path}/{file_name}"
    df = spark.read.csv(path, header=True, inferSchema=True)
    df.write\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable(f"{catalog}.{schema}.{name}_bronze")
    print("saved:", name)

In [0]:
for name in files:
    n = spark.table(f"{catalog}.{schema}.{name}_bronze").count()
    print(name, "rows in bronze:", n)
    spark.table(f"{catalog}.{schema}.{name}_bronze").printSchema()